This notebook takes the various nested directories from the previous notebook and constructs a simple torch dataset. Then we serialize this dataset so that we don't have to perform numerical stacking and disk i/o that will reduce speed.

In [1]:
%load_ext autoreload
%autoreload 2

# Torch Dataset

In [2]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

class DistS1Dataset(Dataset):
    def __init__(self, root_dir=Path('.'), transform=None):
        self.root_dir = Path(root_dir)
        self._parquet_dir = self.root_dir / 'npz_paths'
        self._dataset_dir = self.root_dir / 'dataset_samples_npz'

        # Load and concatenate all Parquet files
        self.df = self._load_parquet_files()

        # Validate the presence of npz_path column
        if 'npz_path' not in self.df.columns:
            raise ValueError("'npz_path' column is required in the parquet files.")

    def _load_parquet_files(self):
        parquet_files = sorted(self._parquet_dir.glob('*.parquet'))
        if not parquet_files:
            raise FileNotFoundError(f"No parquet files found in {str(self.root_dir / self._parquet_dir)}")
        df_list = [pd.read_parquet(pf) for pf in parquet_files]
        df = pd.concat(df_list, ignore_index=True)
        df = df.drop_duplicates().reset_index(drop=True)
        # the paths are relative to the two directories being parallel in the current working directory
        # so we add the root
        df['npz_path'] = f'{str(self.root_dir)}/' + df['npz_path']
        print(f'there were {df.shape[0]:,} samples found')
        return df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        npz_path = row['npz_path']

        # Load the .npz file
        with np.load(npz_path, allow_pickle=False) as npz:
            sample = {key: npz[key] for key in npz.files}

        return sample

In [3]:
%%time

dist_dataset = DistS1Dataset('.')

there were 1,100,655 samples found
CPU times: user 872 ms, sys: 252 ms, total: 1.12 s
Wall time: 3.24 s


# Dataset

## Visualization

In [4]:
for i, data in enumerate(tqdm(dist_dataset)):
    if i > 32:
        break
data['pre_imgs'].shape, data['post_img'].shape, data['acq_dts_float'].shape

  0%|                                  | 33/1100655 [00:06<55:39:13,  5.49it/s]


((14, 2, 256, 256), (2, 256, 256), (15,))

In [5]:
data['acq_dts_float']

array([ 8.23438407,  8.26726078,  8.30013749,  8.33301424,  8.36589098,
        8.39876769,  8.43164447,  9.2206856 ,  9.25356234,  9.28643905,
        9.31931577,  9.35219251,  9.38506925,  9.41794597, 10.43712414])

# Torch Data Loader

In [6]:
import numpy as np
import torch

def left_pad_sequences(sequences, T_max, nodata_value=np.nan):
    batch_size = len(sequences)
    sample_shape = sequences[0].shape[1:] if sequences[0].ndim > 1 else ()

    padded = np.full((batch_size, T_max, *sample_shape), nodata_value, dtype=sequences[0].dtype)
    lengths = np.zeros(batch_size, dtype=np.int64)

    for i, seq in enumerate(sequences):
        T = seq.shape[0]
        lengths[i] = T
        padded[i, T_max - T:] = seq

    return padded, lengths


def custom_collate_fn(batch, T_max=21):
    pre_imgs = [item["pre_imgs"] for item in batch]
    post_img = np.stack([item["post_img"] for item in batch])
    dts = [item['acq_dts_float'] for item in batch]

    padded_pre_imgs, _ = left_pad_sequences(pre_imgs, T_max)
    # dts include the post-img date, so we add one to T_max
    padded_dts, _ = left_pad_sequences(dts, T_max+1)

    
    return {
        "pre_imgs": torch.from_numpy(padded_pre_imgs),
        "post_img": torch.from_numpy(post_img),
        "acq_dts_float": torch.from_numpy(padded_dts)
    }


In [7]:
train_loader = DataLoader(
        dist_dataset,
        batch_size=32,
        shuffle=True,
        collate_fn=custom_collate_fn
    )

In [8]:
dl = iter(train_loader)

In [9]:
batch = next(dl)
batch['pre_imgs'].shape, batch['post_img'].shape, batch['acq_dts_float'].shape

(torch.Size([32, 21, 2, 256, 256]),
 torch.Size([32, 2, 256, 256]),
 torch.Size([32, 22]))